# Context interventions using the frozen predictors

Same-year inputs and outcomes are retained. The frozen evaluation population and predictors are reused; 2015 outcomes are excluded. The intervention runner is under verification and no scientific intervention results should be inferred from the technical checks.


In [1]:
from pathlib import Path
import pandas as pd
from concept_study import measures
from context_intervention.analysis import primary_builds, save_cached_analysis
output = Path('stats/context_intervention')
index = measures.load_index()
pd.DataFrame(primary_builds(index))[['build', 'reference', 'file']]


,build,reference,file
0,all_history,2008,build_all_history_2008.npz
1,all_history,2009,build_all_history_2009.npz
2,last_3,2010,build_last_3_2010.npz
3,last_3,2011,build_last_3_2011.npz
4,last_3,2012,build_last_3_2012.npz
5,last_3,2013,build_last_3_2013.npz


## Cached baseline and error profiles

Calculates yearly prediction metrics and feature, concept and concept–feature profiles by the four prediction groups. Results are saved as compressed CSV files.


In [2]:
tables = save_cached_analysis(output)
tables['baseline_performance']


,build,reference,year,n,deaths,true_positives,false_positives,false_negatives,death_f1,precision,recall,average_precision,auroc,brier,brier_skill
0,all_history,2008,2008,298,22,10,34,12,0.303030,0.227273,0.454545,0.295629,0.782938,0.061466,0.101053
1,all_history,2008,2009,314,23,12,24,11,0.406780,0.333333,0.521739,0.404876,0.860302,0.054413,0.198428
2,all_history,2008,2010,333,24,10,37,14,0.281690,0.212766,0.416667,0.219543,0.703277,0.062572,0.064413
3,all_history,2008,2011,325,41,15,38,26,0.319149,0.283019,0.365854,0.281542,0.674124,0.109356,0.032273
4,all_history,2008,2012,310,28,15,34,13,0.389610,0.306122,0.535714,0.363849,0.820669,0.070783,0.141452
5,all_history,2008,2013,314,27,12,39,15,0.307692,0.235294,0.444444,0.256943,0.707382,0.073221,0.070184
6,all_history,2008,2014,309,19,12,35,7,0.363636,0.255319,0.631579,0.283384,0.845554,0.050449,0.127993
7,all_history,2009,2009,310,27,8,20,19,0.290909,0.285714,0.296296,0.252305,0.770907,0.073911,0.070435
8,all_history,2009,2010,325,26,11,27,15,0.343750,0.289474,0.423077,0.282965,0.820877,0.064051,0.130339
9,all_history,2009,2011,305,32,14,19,18,0.430769,0.424242,0.437500,0.456853,0.799222,0.080066,0.150292


In [3]:
tables['concept_groups'].head()


,build,reference,year,group,concept,n,n_rule,n_active,mean_activation,mean_activation_in_rule,...,activation_rate_in_rule_change,activation_rate_outside_rule_change,rule_rate_change,agreement_rule_active_change,probability_signed_sensitivity_change,probability_absolute_sensitivity_change,probability_tcav_change,log_odds_signed_sensitivity_change,log_odds_absolute_sensitivity_change,log_odds_tcav_change
0,all_history,2008,2008,true_positive,"all_history/2008/2008/42/42:0,135:24,215:233",10,1,5,1.125540,1.548766,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,all_history,2008,2008,true_positive,"all_history/2008/2008/42/42:142,135:265,215:184",10,0,0,0.441739,NaN,...,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
2,all_history,2008,2008,true_positive,"all_history/2008/2008/42/42:218,135:228,215:236",10,5,5,1.906407,2.969074,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,all_history,2008,2008,true_positive,"all_history/2008/2008/42/42:279,135:190,215:167",10,0,0,0.000000,NaN,...,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
4,all_history,2008,2008,true_positive,"all_history/2008/2008/42/42:283,135:131,215:29",10,0,0,0.000000,NaN,...,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0


## Context neighborhoods and development comparisons

The helpers compare feature and concept neighborhoods, then independently validate direct removal, hard selection with sampling, and graded sampling. Group exclusion scores guide selection; they do not identify individual causal effects.


## Selected interventions and temporal evaluation

The full run can take many hours. Enable the flag deliberately after verification. Each update uses earlier development outcomes and evaluates the following year. The cached original, matched random removals, and uniform sampling provide controls.


In [4]:
RUN_FULL_STUDY = False
if RUN_FULL_STUDY:
    from context_intervention.study import run_all
    run_all(output)


## Summaries and interpretation

Compute paired patient-bootstrap intervals after all runs finish. Sampling-seed results are reported separately. Same-year record truncation limits prospective mortality interpretations.


In [5]:
if RUN_FULL_STUDY:
    from context_intervention.report import summarize
    print(summarize(output))


## Additional comparisons

These additions run separately from the original study. Each changes one part of the method relative to a corrected cumulative arm computed by the comparison itself. The corrections: sampled contexts share fixed per-row random keys with the uniform draw (hard and graded draws keep the uniform rows that stay eligible), equal removal priorities are broken by similarity to the representative patients, and the original stable 60/40 patient hash split is preserved. Future outcomes and entrants do not affect assignment. Mortality counts may be uneven. Data roles, the context pool and frozen thresholds are unchanged.

**Latest year:** candidate discovery and validation use only the completed update year. Feature and concept normalization still use the frozen reference statistics.

**Native context states:** extract the labeled context states during context setup, without presenting those examples as prediction queries. The same frozen SAE is applied as an explicit transfer comparison. Spread, active counts, rule membership, outcome/year strata and relative reconstruction error accompany the results. Final context states describe a model endpoint; they are not the per-layer attention keys/values. The original context-as-query arm includes each example's own labeled row in its context and is not leave-one-out. Native retrieval falls back to the original full context when no partially retrieved year/outcome stratum has useful cosine discrimination.

**Selection objective:** compare Death-F1, Brier skill, and AUPRC (average precision) on the same cached validation predictions. Candidate priorities remain F1-based. The cheap comparison launches no new inference; any newly chosen option lacking cached test predictions is marked unavailable. Coverage depends on the selected policy: available results cannot establish objective equivalence or a global ranking. Changed and unchanged choices are counted separately.

**Comparison safeguards:** this added report uses frozen thresholds only; equivalent secondary threshold tuning is unavailable for the new arms. Effective context form, size and fallback status are explicit. Full-context fallbacks are excluded from common-size guided-sampling estimates. Matching random direct controls are reused only where cached; missing controls are reported.


In [6]:
import json
from pathlib import Path
import pandas as pd
comparison_output = Path('stats/context_comparisons')
status_file = comparison_output / 'status.json'
json.loads(status_file.read_text()) if status_file.exists() else {'phase': 'not started'}


{'phase': 'waiting for original study', 'original_pid': 111301}

In [7]:
choices_file = comparison_output / 'metric_choices.csv'
coverage_file = comparison_output / 'objective_coverage.csv'
display(pd.read_csv(coverage_file)) if coverage_file.exists() else print('Coverage pending')
pd.read_csv(choices_file) if choices_file.exists() else 'Metric comparison pending'


,objective,decisions,changed_choices,changed_choices_evaluated,changed_choices_missing,unchanged_choices_evaluated,unchanged_choices_missing,interpretation
0,average_precision,48,25,1,24,23,0,test coverage depends on selected policy; no g...
1,brier_skill,48,24,6,18,24,0,test coverage depends on selected policy; no g...
2,death_f1,48,0,0,0,48,0,test coverage depends on selected policy; no g...


,build,reference,update_year,year,representation,intervention,objective,different_from_death_f1,random_direct_control,evaluation,...,effective_form,context_size,removed_rows,fallback,context_composition,fallback_reason,common_size_eligible,guided_adjustment,secondary_threshold_status,new_model_inferences
0,all_history,2008,2008,2009,features,direct,death_f1,False,not applicable: no direct removal,cached,...,original,3091,0,False,"{""2007/0"": 1751, ""2007/1"": 143, ""2008/0"": 1107...",NaN,False,False,unavailable: equivalent validation tuning not ...,0
1,all_history,2008,2008,2009,features,direct,brier_skill,False,not applicable: no direct removal,cached,...,original,3091,0,False,"{""2007/0"": 1751, ""2007/1"": 143, ""2008/0"": 1107...",NaN,False,False,unavailable: equivalent validation tuning not ...,0
2,all_history,2008,2008,2009,features,direct,average_precision,False,not applicable: no direct removal,cached,...,original,3091,0,False,"{""2007/0"": 1751, ""2007/1"": 143, ""2008/0"": 1107...",NaN,False,False,unavailable: equivalent validation tuning not ...,0
3,all_history,2008,2008,2009,features,hard_sampling,death_f1,False,not applicable: no direct removal,cached,...,uniform,500,0,False,"{""2007/0"": 283, ""2007/1"": 23, ""2008/0"": 179, ""...",NaN,True,False,unavailable: equivalent validation tuning not ...,0
4,all_history,2008,2008,2009,features,hard_sampling,brier_skill,False,not applicable: no direct removal,cached,...,uniform,500,0,False,"{""2007/0"": 283, ""2007/1"": 23, ""2008/0"": 179, ""...",NaN,True,False,unavailable: equivalent validation tuning not ...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,all_history,2009,2010,2011,concepts,hard_sampling,brier_skill,False,not applicable: no direct removal,cached,...,uniform,500,0,False,"{""2007/0"": 158, ""2007/1"": 13, ""2008/0"": 200, ""...",NaN,True,False,unavailable: equivalent validation tuning not ...,0
140,all_history,2009,2010,2011,concepts,hard_sampling,average_precision,False,not applicable: no direct removal,cached,...,uniform,500,0,False,"{""2007/0"": 158, ""2007/1"": 13, ""2008/0"": 200, ""...",NaN,True,False,unavailable: equivalent validation tuning not ...,0
141,all_history,2009,2010,2011,concepts,graded_sampling,death_f1,False,not applicable: no direct removal,cached,...,graded_sampling,500,0,False,"{""2007/0"": 158, ""2007/1"": 13, ""2008/0"": 200, ""...",NaN,True,True,unavailable: equivalent validation tuning not ...,0
142,all_history,2009,2010,2011,concepts,graded_sampling,brier_skill,True,not applicable: no direct removal,cached,...,uniform,500,0,False,"{""2007/0"": 158, ""2007/1"": 13, ""2008/0"": 200, ""...",NaN,True,False,unavailable: equivalent validation tuning not ...,0


The additional inference is queued after the original process finishes. Its log is `stats/context_comparisons/comparisons.log`. Final tables distinguish completed evaluation cells from metric choices that would require extra inference. The native-context comparison retains known outcome labels in context states and does not establish identical concept semantics across roles.
